# Degenerate FWM to coherent coupler transform check

In [172]:
from sympy import *
from time import time

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, C, n2, T'''
)
(delta, nu, Aeff, chi, varsigma) = symbols(
    '''delta, nu, Aeff, chi, varsigma'''
)

gbar2 = Symbol('gbar2', latex_name=r'\bar{g_2}')
gbar3 = Symbol('gbar3', latex_name=r'\bar{g_3}')
gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

esp = Function('esp') # Elementary symmetric polynomials

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

Det = Function('Det') # Unevaluated determinant

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
rhohat = Function('rhohat', latex_name=r'\hat{rho}')

kappa = Function('kappa')
phi = Function('phi')
varphi = Function('varphi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')
wbar = Function('wbar', latex_name=r'\bar{w}')
rhobar = Function('rhobar', latex_name=r'\bar{\rho}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')
rhotilde = Function('rhotilde', latex_name=r'\tilde{\rho}')

A = Function('A')
Ac = Function('Ac')
B = Function('B')
Bc = Function('Bc')

f = Function('f')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
gammabar = IndexedBase('gammabar', latex_name=r'\bar{\gamma}')
gammahat = IndexedBase('gammahat', latex_name=r'\hat{\gamma}')
gammatilde = IndexedBase('gammatilde', latex_name=r'\tilde{\gamma}')
epsilontilde = IndexedBase('epsilontilde', latex_name=r'\tilde{\epsilon}')
ebar = IndexedBase('ebar', latex_name=r'\bar{e}')
etilde = IndexedBase('etilde', latex_name=r'\tilde{e}')
mu = IndexedBase('mu')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')
mutilde = IndexedBase('mutilde', latex_name=r'\tilde{\mu}')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
Theta = IndexedBase('Theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lamda = IndexedBase('lamda') # special sympy spelling
Lamda = IndexedBase('Lamda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


We remind ourselves of some parameter definitions:

In [370]:
b_j_coeffs = [
    Eq(
        b[0], 
        (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
        + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
    ),
    Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2))),
    Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))
]

c_j_coeffs = [Eq(c[0], Product(gamma[j], (j, 1, 2))), Eq(c[1], 0), Eq(c[2], 1)]


d_j_coeffs = [
    Eq(d[0], b[0]**2 - 4*c[0]),
    Eq(d[1], 2*b[0]*b[1] - 4*c[1]),
    Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2]),
    Eq(d[3], 2*b[1]*b[2]),
    Eq(d[4], b[2]**2)
]
d4_lam_0 = Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)
d5_dj = Eq(d[5], d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

## ----


gamma_j_prod_b_poly = Eq(
    Product(gamma[j] -  lamda[1], (j,1,2)), 
    (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4
)
C_gam_prod = Eq(Product(sqrt(gamma[j] - lamda[1]), (j,1,2)), C)
C_gam_prod_sqrd = Eq(C**2, (b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4)
C_gam_prod_sign = Eq(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2, 2*(-1)**m * C)
chi_d5_C = Eq(chi, 16*(-1)**m*C/d[5])
C_d5_chi = Eq(C, solve(chi_d5_C,C)[0])

b_C_one_over_chi = Eq(
    (
        d5_dj.rhs
        .subs([_.args for _ in d_j_coeffs])
        .subs([_.args for _ in c_j_coeffs])
        .subs(b[0], solve(C_gam_prod_sign, b[0])[0])
        /(16*(-1)**m*C)
    )
    .expand(),
    (d5_dj.lhs/(16*(-1)**m*C)).subs([C_d5_chi.args])
)

d5_C_b_lam = Eq(
    d5_dj.lhs,
    d5_dj.rhs
    .subs([_.args for _ in d_j_coeffs])
    .subs([_.args for _ in c_j_coeffs])
    .subs([(b[0], solve(C_gam_prod_sign, b[0])[0])])
    .expand()
    .collect(C, factor)
)



g2_dj = Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)
g3_dj = Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

for _ in b_j_coeffs:
    _
    
for _ in c_j_coeffs:
    _
    
for _ in d_j_coeffs:
    _
d4_lam_0
g2_dj
g3_dj

## ---

gam_sqrt_bj = C_gam_prod.doit().subs(gamma[2], -gamma[1])
gam_sqrt_bj = Eq(
    (2*gam_sqrt_bj.rhs/C_gam_prod_sign.rhs).subs((-1)**(-m), varsigma), 
    2*gam_sqrt_bj.lhs/C_gam_prod_sign.lhs
)

d5_bj_lam = Eq(
    d5_C_b_lam.lhs,
    d5_C_b_lam.rhs.subs(C_gam_prod_sign.rhs/2, C_gam_prod_sign.lhs/2)
    .expand().collect(lamda[1])
)




## --

gamma_j_prod_b_poly
C_gam_prod
C_gam_prod_sqrd
C_gam_prod_sign
chi_d5_C
C_d5_chi
b_C_one_over_chi
d5_C_b_lam
gam_sqrt_bj
d5_bj_lam

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(C, chi*d[5]/(16*(-1)**m))

Eq(b[1]/4 + b[2]*lamda[1]/2 - lamda[1]/(2*(-1)**m*C), 1/chi)

Eq(d[5], 4*(-1)**m*C*(b[1] + 2*b[2]*lamda[1]) - 8*lamda[1])

Eq(varsigma, 2*sqrt(-gamma[1] - lamda[1])*sqrt(gamma[1] - lamda[1])/(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2))

Eq(d[5], (4*b[0]*b[2] + 2*b[1]**2 - 8)*lamda[1] + 2*b[0]*b[1] + 6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3)

The starting point is the following single-parameter degenerate FWM system:

In [326]:
deg_dubar_dvbar = [
    Eq(
        Derivative(ubar(z, mu[1]), z),
        -varsigma*chi*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + 
        chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi
    ),
    Eq(
        Derivative(ubar(z, mu[2]), z),
        -varsigma*chi*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + 
        chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi
    ),
    Eq(
        Derivative(ubar(z, mu[3]), z),
        -varsigma*chi*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + 
        chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[1]), z),
        varsigma*chi*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - 
        chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[2]), z),
        varsigma*chi*ubar(z, mu[1])*ubar(z, mu[3])**2/4 - 
        chi*ubar(z, mu[2])*vbar(z, mu[2])**2/2 + vbar(z, mu[2])/chi
    ),
    Eq(
        Derivative(vbar(z, mu[3]), z),
        varsigma*chi*ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])/2 - 
        chi*ubar(z, mu[3])*vbar(z, mu[3])**2/4 + vbar(z, mu[3])/chi
    )
]
deg_dubar_dvbar_subs = [_.args for _ in deg_dubar_dvbar]

H_deg_ubar_vbar = Eq(
    Hbar,
    -varsigma*chi*(
        ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + 
        vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2
    )/4 
    + chi*(
        2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 
        2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + 
        ubar(z, mu[3])**2*vbar(z, mu[3])**2
    )/8 
    - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi
)

assert diff(H_deg_ubar_vbar.rhs.doit(),z).subs(deg_dubar_dvbar_subs).simplify() == 0, 'ham not conserved'


## Intermodal power

eps_vals = [
    Eq(epsilon[1], Rational(3/4)),
    Eq(epsilon[2], Rational(3/4)),
    Eq(epsilon[3], Rational(3/2))
]
eps_vals_subs = [_.args for _ in eps_vals]

rhobar_uvs = [
    Eq(
        ubar(z, mu[_j+1])*vbar(z, mu[_j+1]), gammabar[_j+1] - eps_vals_subs[_j][1]*rhobar(z)
    ).subs(_eps_sols) 
    for _j in range(3)
]
rhobar_uv_subs = [_.args for _ in rhobar_uvs]

gammabar_sum = Eq(Sum(gammabar[j], (j,1,3)), 0)
rhobar_mean = Eq(
    -(sum(_.rhs for _ in rhobar_uvs)/3).subs(gammabar[3], solve(gammabar_sum.doit(), gammabar[3])[0]), 
    -sum(_.lhs for _ in rhobar_uvs)/3
)

_gambar_diffs = [
    ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]),
    ubar(z, mu[1])*vbar(z, mu[1])*2 - ubar(z, mu[3])*vbar(z, mu[3]),
    ubar(z, mu[2])*vbar(z, mu[2])*2 - ubar(z, mu[3])*vbar(z, mu[3])
]

gambar_pow_cons = [
    Eq(_, _.subs(rhobar_uv_subs)) for _ in _gambar_diffs
]

for _ in gambar_pow_cons:
    assert diff(_.lhs,z).doit().subs(deg_dubar_dvbar_subs).factor() == 0, 'intermodal power not conserved'



H_deg_ubar_vbar
print()
for _ in deg_dubar_dvbar:
    _
    
print()
gammabar_sum
rhobar_mean
for _ in rhobar_uvs:
    _
for _ in gambar_pow_cons:
    _

Eq(Hbar, -chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4 + chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(ubar(z, mu[1]), z), -chi*varsigma*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi)

Eq(Derivative(ubar(z, mu[2]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi)

Eq(Derivative(ubar(z, mu[3]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi)

Eq(Derivative(vbar(z, mu[1]), z), chi*varsigma*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi)

...

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -3*rhobar(z)/4 + gammabar[2])

Eq(ubar(z, mu[3])*vbar(z, mu[3]), -3*rhobar(z)/2 + gammabar[3])

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), gammabar[1] - gammabar[2])

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[1] - gammabar[3])

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[2] - gammabar[3])

Elsehwere we derived the below value for $\bar{H}$ but we hope that this will be found by another approach because previously we relied on the form of the analytic solutions to get it:

In [325]:
# Val if maps to coherent coupler
H_deg_ubar_vbar_val = Eq(Hbar, -b[2]*d[5]/8 - 2/chi + 64*(d[5] + 16*lamda[1])*lamda[1]/(chi**3*d[5]**2))
H_deg_ubar_vbar_val

Eq(Hbar, -b[2]*d[5]/8 - 2/chi + (64*d[5] + 1024*lamda[1])*lamda[1]/(chi**3*d[5]**2))

The target system we hope to transform the degenerate FWM system into is the following coherent coupler system:

In [153]:
Hhat_uv_b = Eq(
    -(gamma[1] - gamma[2])**2*b[2]/4 + b[0],
    uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + 
    Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2))
)

uvhat_rho = [
    Eq(uhat(z, mu[1])*vhat(z, mu[1]), gamma[1] - rho(z)),
    Eq(uhat(z, mu[2])*vhat(z, mu[2]), gamma[2] - rho(z))
]
uvhat_rho_subs = [_.args for _ in uvhat_rho]

uvhat_rho_ham_target = Eq(
    (Hhat_uv_b.lhs - Hhat_uv_b.rhs.doit())
    .subs(uvhat_rho_subs)
    .subs(gamma[2], - gamma[1])
    .expand().collect(rho(z), factor)
    , 
    0
)

Hhat_uv_b
uvhat_rho_ham_target
for _ in uvhat_rho:
    _

Eq(-(gamma[1] - gamma[2])**2*b[2]/4 + b[0], uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) + Sum(-uhat(z, mu[j])**2*vhat(z, mu[j])**2*b[2]/2 + uhat(z, mu[j])*vhat(z, mu[j])*b[1]/2, (j, 1, 2)))

Eq(rho(z)**2*b[2] + rho(z)*b[1] - uhat(z, mu[1])*uhat(z, mu[2]) - vhat(z, mu[1])*vhat(z, mu[2]) + b[0], 0)

Eq(uhat(z, mu[1])*vhat(z, mu[1]), -rho(z) + gamma[1])

Eq(uhat(z, mu[2])*vhat(z, mu[2]), -rho(z) + gamma[2])

In [104]:
gamma_lamda_bj_gam1 = Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)
gamma_lamda_bj_gam1

Eq(gamma[1]**2, -(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2/4 + lamda[1]**2)

We believe, and want to prove, that the transform is given by:

In [66]:
u_v_hat_as_u_v_bar = [
    Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])*exp(z*theta)/vbar(z, mu[3])),
    Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*exp(-z*theta)/vbar(z, mu[3])),
    Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])*exp(-z*theta)/ubar(z, mu[3])),
    Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*exp(z*theta)/ubar(z, mu[3]))
]
u_v_hat_as_u_v_bar_subs = [_.args for _ in u_v_hat_as_u_v_bar]

for _ in u_v_hat_as_u_v_bar: 
    _

Eq(uhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*ubar(z, mu[1])*exp(z*theta)/vbar(z, mu[3]))

Eq(uhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*ubar(z, mu[2])*exp(-z*theta)/vbar(z, mu[3]))

Eq(vhat(z, mu[1]), sqrt(2)*sqrt(gamma[1] - lamda[1])*vbar(z, mu[1])*exp(-z*theta)/ubar(z, mu[3]))

Eq(vhat(z, mu[2]), sqrt(2)*sqrt(gamma[2] - lamda[1])*vbar(z, mu[2])*exp(z*theta)/ubar(z, mu[3]))

In [328]:
_uvhat_12_term = uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2])
uv_12_diff_hat = Eq(
    _uvhat_12_term, 
    _uvhat_12_term.subs(uvhat_rho_subs)
)

bars_to_3 = [
    Eq(ubar(z, mu[1])*vbar(z, mu[1]), solve(gambar_pow_cons[1],ubar(z, mu[1])*vbar(z, mu[1]))[0]),
    Eq(ubar(z, mu[2])*vbar(z, mu[2]), solve(gambar_pow_cons[2],ubar(z, mu[2])*vbar(z, mu[2]))[0])
]
bars_to_3_subs = [_.args for _ in bars_to_3]
uv_12_diff_hat_bar3 = uv_12_diff_hat.subs(u_v_hat_as_u_v_bar_subs).subs(bars_to_3_subs)
uv_12_diff_hat_bar3 = Eq( 
    uv_12_diff_hat_bar3.lhs.expand().collect(ubar(z,mu[3])*vbar(z,mu[3])),
    uv_12_diff_hat_bar3.rhs
)
gammabar_2_as_1 = Eq(
    gammabar[2], 
    solve(
        uv_12_diff_hat_bar3.subs([
        (gamma[2], -gamma[1]),
        (gammabar[3], -gammabar[1]-gammabar[2])
        ]).simplify(), 
        gammabar[2]
    )[0]
)
gamma_lam_as_gammabar = Eq(gamma[1]/lamda[1], solve(gammabar_2_as_1.subs(gamma[1], x*lamda[1]),x)[0])

uv_12_diff_hat
uv_12_diff_hat_bar3
gammabar_2_as_1
gamma_lam_as_gammabar

Eq(uhat(z, mu[1])*vhat(z, mu[1]) - uhat(z, mu[2])*vhat(z, mu[2]), gamma[1] - gamma[2])

Eq((2*gamma[1]*gammabar[1] - gamma[1]*gammabar[3] - 2*gamma[2]*gammabar[2] + gamma[2]*gammabar[3] - 2*gammabar[1]*lamda[1] + 2*gammabar[2]*lamda[1])/(ubar(z, mu[3])*vbar(z, mu[3])) + gamma[1] - gamma[2], gamma[1] - gamma[2])

Eq(gammabar[2], (-2*gamma[1] + lamda[1])*gammabar[1]/(2*gamma[1] + lamda[1]))

Eq(gamma[1]/lamda[1], (gammabar[1] - gammabar[2])/(2*(gammabar[1] + gammabar[2])))

In [329]:
_uvhat_12_gams_as_ubar3_term = (
    uhat(z, mu[2])*vhat(z, mu[2])*(gamma[1] - lamda[1]) -
    uhat(z, mu[1])*vhat(z, mu[1])*(gamma[2] - lamda[1])
)

uvhat_12_gams_as_ubar3 = Eq(
    _uvhat_12_gams_as_ubar3_term,
    _uvhat_12_gams_as_ubar3_term
    .subs(u_v_hat_as_u_v_bar_subs)
    .factor()
    .subs([gambar_pow_cons[0].args])
)
delta_gam_gambar = Eq(delta, fraction(uvhat_12_gams_as_ubar3.rhs)[0])

ubar3_as_uvhat_12_gams = Eq(
    (delta_gam_gambar.rhs/uvhat_12_gams_as_ubar3.rhs),
    (delta_gam_gambar.lhs/uvhat_12_gams_as_ubar3.lhs)
)

uvhat_12_gams_as_ubar3
delta_gam_gambar
ubar3_as_uvhat_12_gams

Eq((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]), -2*(gamma[1] - lamda[1])*(gamma[2] - lamda[1])*(gammabar[1] - gammabar[2])/(ubar(z, mu[3])*vbar(z, mu[3])))

Eq(delta, -2*(gamma[1] - lamda[1])*(gamma[2] - lamda[1])*(gammabar[1] - gammabar[2]))

Eq(ubar(z, mu[3])*vbar(z, mu[3]), delta/((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1])))

In [330]:
ubar_vbar_from_u_v_hat = [
    Eq(ubar(z,mu[1]), solve(u_v_hat_as_u_v_bar[0], ubar(z,mu[1]))[0]),
    Eq(ubar(z,mu[2]), solve(u_v_hat_as_u_v_bar[1], ubar(z,mu[2]))[0]),
    Eq(vbar(z,mu[1]), solve(u_v_hat_as_u_v_bar[2], vbar(z,mu[1]))[0]),
    Eq(vbar(z,mu[2]), solve(u_v_hat_as_u_v_bar[3], vbar(z,mu[2]))[0])
]
ubar_vbar_from_u_v_hat_subs = [_.args for _ in ubar_vbar_from_u_v_hat]

for _ in u_v_hat_as_u_v_bar:
    Eq(
        diff(_.lhs,z), 
        diff(_.rhs,z)
        .subs(deg_dubar_dvbar_subs)
        .subs(ubar_vbar_from_u_v_hat_subs)
        .expand().collect([varsigma, ubar(z, mu[3])*vbar(z, mu[3])], factor)
        .subs([ubar3_as_uvhat_12_gams.args])
    )

Eq(Derivative(uhat(z, mu[1]), z), -chi*delta*varsigma*(uhat(z, mu[1])**2*uhat(z, mu[2]) + vhat(z, mu[2])*gamma[1] - vhat(z, mu[2])*lamda[1])/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) + chi*delta*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*uhat(z, mu[1])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[1] - 4*lamda[1])) + (chi*theta - 2)*uhat(z, mu[1])/chi)

Eq(Derivative(uhat(z, mu[2]), z), -chi*delta*varsigma*(uhat(z, mu[1])*uhat(z, mu[2])**2 + vhat(z, mu[1])*gamma[2] - vhat(z, mu[1])*lamda[1])/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) + chi*delta*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*uhat(z, mu[2])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[2] - 4*lamda[1])) - (chi*theta + 2)*uhat(z, mu[2])/chi)

Eq(Derivative(vhat(z, mu[1]), z), chi*delta*varsigma*(uhat(z, mu[2])*gamma[1] - uhat(z, mu[2])*lamda[1] + vhat(z, mu[1])**2*vhat(z, mu[2]))/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) - chi*delta*(uhat(z, mu[1])*vhat(z, mu[1]) + gamma[1] - lamda[1])*vhat(z, mu[1])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[1] - 4*lamda[1])) - (chi*theta - 2)*vhat(z, mu[1])/chi)

Eq(Derivative(vhat(z, mu[2]), z), chi*delta*varsigma*(uhat(z, mu[1])*gamma[2] - uhat(z, mu[1])*lamda[1] + vhat(z, mu[1])*vhat(z, mu[2])**2)/(4*((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])) - chi*delta*(uhat(z, mu[2])*vhat(z, mu[2]) + gamma[2] - lamda[1])*vhat(z, mu[2])/(((gamma[1] - lamda[1])*uhat(z, mu[2])*vhat(z, mu[2]) - (gamma[2] - lamda[1])*uhat(z, mu[1])*vhat(z, mu[1]))*(4*gamma[2] - 4*lamda[1])) + (chi*theta + 2)*vhat(z, mu[2])/chi)

We will make two choices. First, we fix a value for $\bar{\gamma}_1 - \bar{\gamma}_1$, then we fix a value for $\chi$. We note that our choice for $\bar{\gamma}_1 - \bar{\gamma}_1$ is not inconsistent with the map:

In [407]:
gambar_dif_choice = Eq(
    gammabar[1] - gammabar[2], 
    gamma[1]*d[5]/(4*(gamma[1]**2 - lamda[1]**2))
)
gammabar_gamma_subs = solve([gamma_lam_as_gammabar, gambar_dif_choice], [gammabar[1], gammabar[2]])

# Define d6
Eq(d[6], d[5]/(8*(gamma[1]**2 - lamda[1]**2)))
Eq(
    d[6],
    -(
        2*(2*b[0]*b[2] + b[1]**2 - 4)*lamda[1] + 2*b[0]*b[1] + 
        6*b[1]*b[2]*lamda[1]**2 + 4*b[2]**2*lamda[1]**3
    )/(2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2)
)

# These are choices we know work but they came from the wp solutions so here we say that we choose them
# (note consistent with derived expression for gamma[1]/lambda[1])
Eq(gamma[1], (gammabar[1] - gammabar[2])/(2*d[6]))
Eq(lamda[1], (gammabar[1] + gammabar[2])/d[6])

gambar_dif_choice

for _k, _v in gammabar_gamma_subs.items():
    Eq(_k, _v)

Eq(d[6], d[5]/(8*gamma[1]**2 - 8*lamda[1]**2))

Eq(d[6], (-(4*b[0]*b[2] + 2*b[1]**2 - 8)*lamda[1] - 2*b[0]*b[1] - 6*b[1]*b[2]*lamda[1]**2 - 4*b[2]**2*lamda[1]**3)/(2*(b[0] + b[1]*lamda[1] + b[2]*lamda[1]**2)**2))

Eq(gamma[1], (gammabar[1] - gammabar[2])/(2*d[6]))

Eq(lamda[1], (gammabar[1] + gammabar[2])/d[6])

Eq(gammabar[1] - gammabar[2], d[5]*gamma[1]/(4*gamma[1]**2 - 4*lamda[1]**2))

Eq(gammabar[1], (2*gamma[1] + lamda[1])*d[5]/(16*(gamma[1]**2 - lamda[1]**2)))

Eq(gammabar[2], (-2*gamma[1] + lamda[1])*d[5]/(16*(gamma[1]**2 - lamda[1]**2)))

In [369]:
delta_d5 = delta_gam_gambar.subs([gambar_dif_choice.args, (gamma[2], -gamma[1])]).simplify()

chi_delta = chi_d5_C.subs([
    (C_gam_prod.rhs, C_gam_prod.lhs), 
    ((-1)**m, varsigma), 
    (d[5], solve(delta_d5, d[5])[0])
]).doit()

chi_d5_sqrt_gam = chi_delta.subs([delta_d5.args])

delta_d5
chi_delta
chi_d5_sqrt_gam

Eq(delta, d[5]*gamma[1]/2)

Eq(chi, 8*varsigma*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])*gamma[1]/delta)

Eq(chi, 16*varsigma*sqrt(gamma[1] - lamda[1])*sqrt(gamma[2] - lamda[1])/d[5])

We confirm that the DFWM Hamiltonian then transforms into the target Hamiltonian for the coherent coupler. We have assumed the value for $\bar{H}$ derived elsewhere:

In [366]:
H_deg_to_rho_hat = Eq(
    (
        (
            H_deg_ubar_vbar.lhs
            .subs([H_deg_ubar_vbar_val.args])
            - H_deg_ubar_vbar.rhs
            .doit()
            .subs(ubar_vbar_from_u_v_hat_subs)
        )/(ubar(z, mu[3])*vbar(z, mu[3]))**2/(2/d[5])
    )
    .expand().collect(ubar(z, mu[3])*vbar(z, mu[3]), factor)
    .subs([
        ubar3_as_uvhat_12_gams.args, 
        chi_delta.args,
        gam_sqrt_bj.args,
        delta_d5.args
    ])
    .subs(uvhat_rho_subs)
    .subs([
        (gamma[2], -gamma[1])
    ])
    .expand().collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    .subs([gamma_lamda_bj_gam1.args])
    .collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    .subs([d5_bj_lam.args])
    .collect([rho(z), uhat(z, mu[1]), vhat(z, mu[1])], simplify)
    ,
    0
)

assert uvhat_rho_ham_target.subs(b[0], solve(H_deg_to_rho_hat,b[0])[0]), 'target ham not obtained'

H_deg_to_rho_hat

Eq(-rho(z)**2*b[2] - rho(z)*b[1] + uhat(z, mu[1])*uhat(z, mu[2]) + vhat(z, mu[1])*vhat(z, mu[2]) - b[0], 0)

## Solving the DFWM system

In [372]:
H_deg_ubar_vbar
print()
for _ in deg_dubar_dvbar:
    _
    
print()
gammabar_sum
rhobar_mean
for _ in rhobar_uvs:
    _
for _ in gambar_pow_cons:
    _

Eq(Hbar, -chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4 + chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 - Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(ubar(z, mu[1]), z), -chi*varsigma*vbar(z, mu[2])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[1])**2*vbar(z, mu[1])/2 - ubar(z, mu[1])/chi)

Eq(Derivative(ubar(z, mu[2]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[3])**2/4 + chi*ubar(z, mu[2])**2*vbar(z, mu[2])/2 - ubar(z, mu[2])/chi)

Eq(Derivative(ubar(z, mu[3]), z), -chi*varsigma*vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])/2 + chi*ubar(z, mu[3])**2*vbar(z, mu[3])/4 - ubar(z, mu[3])/chi)

Eq(Derivative(vbar(z, mu[1]), z), chi*varsigma*ubar(z, mu[2])*ubar(z, mu[3])**2/4 - chi*ubar(z, mu[1])*vbar(z, mu[1])**2/2 + vbar(z, mu[1])/chi)

...

Eq(ubar(z, mu[2])*vbar(z, mu[2]), -3*rhobar(z)/4 + gammabar[2])

Eq(ubar(z, mu[3])*vbar(z, mu[3]), -3*rhobar(z)/2 + gammabar[3])

Eq(ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[2])*vbar(z, mu[2]), gammabar[1] - gammabar[2])

Eq(2*ubar(z, mu[1])*vbar(z, mu[1]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[1] - gammabar[3])

Eq(2*ubar(z, mu[2])*vbar(z, mu[2]) - ubar(z, mu[3])*vbar(z, mu[3]), 2*gammabar[2] - gammabar[3])

In [393]:
wm_deg_ubar_vbar = Eq(
    H_deg_ubar_vbar.rhs - H_deg_ubar_vbar.rhs.subs(varsigma,0),
    H_deg_ubar_vbar.lhs - H_deg_ubar_vbar.rhs.subs(varsigma,0)
)

duvbar1 = Eq(
    Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z),
    Derivative(ubar(z, mu[1])*vbar(z, mu[1]),z)
    .doit()
    .subs(deg_dubar_dvbar_subs)
    .factor()
)

drhobar_sqrd = Eq(
    (duvbar1.lhs**2)
    .subs(rhobar_uv_subs)
    .doit()*Rational(16,9),
    (((duvbar1.rhs**2) - wm_deg_ubar_vbar.lhs**2 + wm_deg_ubar_vbar.rhs**2)*Rational(16,9))
    .doit().expand()
    .subs(rhobar_uv_subs)
    .expand().collect(rhobar(z), simplify)
    .subs([
        (varsigma**2, 1),
        (gammabar[3], solve(gammabar_sum.doit(), gammabar[3])[0])
    ])
    .collect(rhobar(z), simplify)
)

duvbar1
wm_deg_ubar_vbar
drhobar_sqrd

Eq(Derivative(ubar(z, mu[1])*vbar(z, mu[1]), z), chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 - vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4)

Eq(-chi*varsigma*(ubar(z, mu[1])*ubar(z, mu[2])*ubar(z, mu[3])**2 + vbar(z, mu[1])*vbar(z, mu[2])*vbar(z, mu[3])**2)/4, Hbar - chi*(2*ubar(z, mu[1])**2*vbar(z, mu[1])**2 + 2*ubar(z, mu[2])**2*vbar(z, mu[2])**2 + ubar(z, mu[3])**2*vbar(z, mu[3])**2)/8 + Sum(ubar(z, mu[j])*vbar(z, mu[j]), (j, 1, 3))/chi)

Eq(Derivative(rhobar(z), z)**2, 16*Hbar**2/9 - 4*Hbar*chi*gammabar[1]**2/3 - 8*Hbar*chi*gammabar[1]*gammabar[2]/9 - 4*Hbar*chi*gammabar[2]**2/3 + chi**2*gammabar[1]**4/4 - chi**2*gammabar[1]**3*gammabar[2]/9 - 5*chi**2*gammabar[1]**2*gammabar[2]**2/18 - chi**2*gammabar[1]*gammabar[2]**3/9 + chi**2*gammabar[2]**4/4 + 6*rhobar(z)**3 + (-32*Hbar + chi*(chi**2*gammabar[1]**3 - chi**2*gammabar[1]**2*gammabar[2] - chi**2*gammabar[1]*gammabar[2]**2 + chi**2*gammabar[2]**3 + 12*gammabar[1]**2 + 8*gammabar[1]*gammabar[2] + 12*gammabar[2]**2))*rhobar(z)/(3*chi) + (chi**3*(-4*Hbar + 3*chi*gammabar[1]**2 + 2*chi*gammabar[1]*gammabar[2] + 3*chi*gammabar[2]**2) + 32)*rhobar(z)**2/(2*chi**2))

In [411]:
__xx = wm_deg_ubar_vbar.rhs.doit().subs(rhobar_uv_subs).subs([
        (varsigma**2, 1),
        (gammabar[3], solve(gammabar_sum.doit(), gammabar[3])[0])
    ]).subs(gammabar_gamma_subs).expand().collect([rhobar(z), Hbar], factor)

In [412]:
__xx

Hbar - 9*chi*rhobar(z)**2/16 - chi*(2*gamma[1]**2 + lamda[1]**2)*d[5]**2/(256*(gamma[1] - lamda[1])**2*(gamma[1] + lamda[1])**2) - 3*rhobar(z)/chi

In [415]:
drhobar_sqrd.rhs.subs(gammabar_gamma_subs).expand().collect([Hbar, rhobar(z)], factor)

16*Hbar**2/9 - Hbar*(144*chi**2*rhobar(z)**2*gamma[1]**4 - 288*chi**2*rhobar(z)**2*gamma[1]**2*lamda[1]**2 + 144*chi**2*rhobar(z)**2*lamda[1]**4 + 2*chi**2*d[5]**2*gamma[1]**2 + chi**2*d[5]**2*lamda[1]**2 + 768*rhobar(z)*gamma[1]**4 - 1536*rhobar(z)*gamma[1]**2*lamda[1]**2 + 768*rhobar(z)*lamda[1]**4)/(72*chi*(gamma[1] - lamda[1])**2*(gamma[1] + lamda[1])**2) + chi**2*(gamma[1]**2 + 2*lamda[1]**2)*d[5]**4*gamma[1]**2/(9216*(gamma[1] - lamda[1])**4*(gamma[1] + lamda[1])**4) + 6*rhobar(z)**3 + (chi**2*d[5]*gamma[1]**2*lamda[1] + 32*gamma[1]**4 - 16*gamma[1]**2*lamda[1]**2 - 16*lamda[1]**4)*rhobar(z)*d[5]**2/(384*(gamma[1] - lamda[1])**3*(gamma[1] + lamda[1])**3) + (2*chi**4*d[5]**2*gamma[1]**2 + chi**4*d[5]**2*lamda[1]**2 + 1024*gamma[1]**4 - 2048*gamma[1]**2*lamda[1]**2 + 1024*lamda[1]**4)*rhobar(z)**2/(64*chi**2*(gamma[1] - lamda[1])**2*(gamma[1] + lamda[1])**2)